# Titanic 
## Score: .77511


In [20]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    XGBClassifier = None
    HAS_XGB = False

SEED = 42
N_SPLITS = 7
np.random.seed(SEED)
DATA_ROOT = Path.cwd() / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}


In [21]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    for pcl in (1, 2, 3):
        m = tr.loc[tr["Pclass"] == pcl, "Fare"].median()
        if pd.notna(m):
            tr.loc[tr["Pclass"] == pcl, "Fare"] = tr.loc[tr["Pclass"] == pcl, "Fare"].fillna(m)
            te.loc[te["Pclass"] == pcl, "Fare"] = te.loc[te["Pclass"] == pcl, "Fare"].fillna(m)
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["Deck"] = out["Cabin"].apply(
            lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U"
        )
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeBin"] = pd.cut(
            out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]
        ).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    return feats(tr), y, feats(te)


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits, alpha: float):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    gmean = y_s.mean()
    Xn = X.copy()
    Xtn = X_test.copy()
    for col in cols:
        oof = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            sm = (stat["mean"] * stat["count"] + gmean * alpha) / (stat["count"] + alpha)
            oof[va_i] = X.iloc[va_i][col].map(sm).fillna(gmean).to_numpy()
        full = y_s.groupby(X[col]).agg(["mean", "count"])
        smf = (full["mean"] * full["count"] + gmean * alpha) / (full["count"] + alpha)
        Xn[col + "_te"] = oof
        Xtn[col + "_te"] = X_test[col].map(smf).fillna(gmean).to_numpy()
    return Xn, Xtn


def build_matrix(df: pd.DataFrame) -> np.ndarray:
    emb = {"S": 0.0, "C": 1.0, "Q": 2.0}
    d = df.copy()
    d["SexF"] = (d["Sex"] == "female").astype(np.float64)
    d["Pc"] = pd.to_numeric(d["Pclass"], errors="coerce").fillna(3).astype(np.float64)
    d["EmbN"] = d["Embarked"].map(emb).fillna(0.0)
    num = [
        "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone",
        "TicketGroupSize", "FarePerPerson", "HasCabin", "SexF", "Pc", "EmbN",
    ]
    te_cols = [c for c in d.columns if c.endswith("_te")]
    use = num + sorted(te_cols)
    M = d[use].replace([np.inf, -np.inf], np.nan)
    return M.to_numpy(dtype=np.float64), use


train_raw = pd.read_csv(DATA_ROOT / "train.csv")
test_raw = pd.read_csv(DATA_ROOT / "test.csv")
pid = test_raw["PassengerId"].to_numpy()

X, y, X_test = build_xy(train_raw, test_raw)
TE_COLS = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, TE_COLS, n_splits=N_SPLITS, alpha=22.0)

M_tr, COLS = build_matrix(X_te)
M_te, _ = build_matrix(X_test_te)
imp = SimpleImputer(strategy="median")
X_all = imp.fit_transform(M_tr)
X_sub = imp.transform(M_te)
print("matrix", X_all.shape, "HAS_XGB", HAS_XGB)


matrix (891, 19) HAS_XGB True


In [22]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_rf = np.zeros(len(y))
oof_hgb = np.zeros(len(y))
p_rf_t = np.zeros(len(pid))
p_hgb_t = np.zeros(len(pid))

if HAS_XGB:
    oof_xgb = np.zeros(len(y))
    p_xgb_t = np.zeros(len(pid))

for fold, (tr_i, va_i) in enumerate(cv.split(X_all, y), start=1):
    Xtr, Xva = X_all[tr_i], X_all[va_i]
    ytr, yva = y[tr_i], y[va_i]

    if HAS_XGB:
        xgb = XGBClassifier(
            n_estimators=1200,
            learning_rate=0.028,
            max_depth=4,
            min_child_weight=2,
            subsample=0.88,
            colsample_bytree=0.75,
            reg_lambda=2.0,
            reg_alpha=0.1,
            random_state=SEED + fold,
            eval_metric="logloss",
            early_stopping_rounds=60,
            n_jobs=-1,
        )
        xgb.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        oof_xgb[va_i] = xgb.predict_proba(Xva)[:, 1]
        p_xgb_t += xgb.predict_proba(X_sub)[:, 1] / cv.n_splits

    rf = RandomForestClassifier(
        n_estimators=650,
        max_depth=14,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=SEED + 17 * fold,
        n_jobs=-1,
    )
    rf.fit(Xtr, ytr)
    oof_rf[va_i] = rf.predict_proba(Xva)[:, 1]
    p_rf_t += rf.predict_proba(X_sub)[:, 1] / cv.n_splits

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.045,
        max_iter=900,
        max_depth=7,
        min_samples_leaf=8,
        l2_regularization=0.55,
        random_state=SEED + 31 * fold,
    )
    hgb.fit(Xtr, ytr)
    oof_hgb[va_i] = hgb.predict_proba(Xva)[:, 1]
    p_hgb_t += hgb.predict_proba(X_sub)[:, 1] / cv.n_splits

    print(
        "fold",
        fold,
        "va_rf",
        round(accuracy_score(yva, rf.predict(Xva)), 4),
        "va_hgb",
        round(accuracy_score(yva, hgb.predict(Xva)), 4),
    )

if HAS_XGB:
    stack_oof = np.column_stack([oof_xgb, oof_rf, oof_hgb])
else:
    stack_oof = np.column_stack([oof_rf, oof_hgb])

meta = LogisticRegression(max_iter=3000, C=0.35, random_state=SEED)
meta.fit(stack_oof, y)
m_oof = meta.predict_proba(stack_oof)[:, 1]

best_a, best_t = -1.0, 0.5
for t in np.linspace(0.28, 0.72, 221):
    a = accuracy_score(y, (m_oof >= t).astype(int))
    if a > best_a:
        best_a, best_t = a, float(t)

print("cv_meta_oof_acc", round(best_a, 5), "t", round(best_t, 3))
print(confusion_matrix(y, (m_oof >= best_t).astype(int)))
print("meta_coef", np.round(meta.coef_, 4), "intercept", round(float(meta.intercept_[0]), 4))


fold 1 va_rf 0.875 va_hgb 0.8672
fold 2 va_rf 0.8438 va_hgb 0.875
fold 3 va_rf 0.8268 va_hgb 0.7795
fold 4 va_rf 0.8268 va_hgb 0.8031
fold 5 va_rf 0.8425 va_hgb 0.8425
fold 6 va_rf 0.811 va_hgb 0.811
fold 7 va_rf 0.8425 va_hgb 0.8504
cv_meta_oof_acc 0.85073 t 0.386
[[486  63]
 [ 70 272]]
meta_coef [[1.8234 2.1932 1.0247]] intercept -2.5365


In [23]:
if HAS_XGB:
    stack_te = np.column_stack([p_xgb_t, p_rf_t, p_hgb_t])
else:
    stack_te = np.column_stack([p_rf_t, p_hgb_t])
blend = meta.predict_proba(stack_te)[:, 1]
submission = pd.DataFrame({"PassengerId": pid, "Survived": (blend >= best_t).astype(int)})
submission.to_csv(Path.cwd() / "submission.csv", index=False)
print("saved", Path.cwd() / "submission.csv")
submission.head(12)


saved c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
